# AURA · Demostración de avatar neuronal

Animá el retrato sintético de **AURA** con SadTalker, una red neuronal que genera labios y movimientos faciales a partir de audio. **Esta prueba es un video pregrabado, no una videollamada en tiempo real.**

1. En Colab: **Entorno de ejecución → Cambiar tipo de entorno → T4 GPU** (según disponibilidad gratuita).
2. Ejecutá las celdas en orden o seleccioná **Ejecutar todo**.
3. La última celda mostrará el MP4 y permitirá descargarlo.

No requiere claves de API ni habilitar facturación. Los programas y modelos externos se descargarán durante la ejecución. **Aún no probamos la inferencia en una GPU real**; SadTalker usa dependencias antiguas y puede exigir ajustes.

In [ ]:
import subprocess
p=subprocess.run(['nvidia-smi'],capture_output=True,text=True)
assert p.returncode==0, 'Para este cuaderno necesitás seleccionar GPU T4 en Colab (si está disponible).'
print(p.stdout[:2000])

In [ ]:
import subprocess,os
from pathlib import Path
subprocess.run('apt-get update -qq && apt-get install -y -qq ffmpeg git wget',shell=True,check=True)
if not Path('/content/SadTalker/inference.py').exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/OpenTalker/SadTalker.git','/content/SadTalker'],check=True)
print('SadTalker preparado')

In [ ]:
import requests,re,base64,subprocess
subprocess.run(['python','-m','pip','install','-q','gTTS'],check=True)
from pathlib import Path
from PIL import Image
from gtts import gTTS
folder=Path('/content/AURA_media');folder.mkdir(exist_ok=True)
html=requests.get('https://aura-iphone-v07.onrender.com/',timeout=60).text
match=re.search(r'femenino-0\.webp"\s*:\s*"data:image/webp;base64,([A-Za-z0-9+/=]+)"',html)
assert match, 'No encontré el avatar en la versión publicada de AURA. Usá el paquete descargable como alternativa.'
(folder/'aura_source.webp').write_bytes(base64.b64decode(match.group(1)))
Image.open(folder/'aura_source.webp').convert('RGB').save(folder/'aura_face.png')
gTTS('Hola, soy AURA. Estoy lista para hablar con vos, ayudarte en tus proyectos y acompañarte cada día.',lang='es',tld='com.ar').save(str(folder/'voice.mp3'))
subprocess.run(['ffmpeg','-y','-loglevel','error','-i',str(folder/'voice.mp3'),'-ar','16000','-ac','1',str(folder/'voice.wav')],check=True)
from IPython.display import display,Audio
display(Image.open(folder/'aura_face.png').resize((240,320)))
display(Audio(filename=str(folder/'voice.wav')))
print('Retrato y audio preparados')

## Dependencias

Se crea un entorno aislado Python 3.8 porque SadTalker no está actualizado para algunas versiones modernas del Python de Colab. Si Colab cambia sus versiones o la red bloquea las descargas, la instalación podría fallar.

In [ ]:
import subprocess
setup=r'''set -e
export CONDA_PLUGINS_AUTO_ACCEPT_TOS=true
if [ ! -x /content/miniconda3/bin/conda ]; then
 wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh -O /tmp/miniconda.sh
 bash /tmp/miniconda.sh -b -p /content/miniconda3
fi
source /content/miniconda3/bin/activate
if ! conda env list | grep -q '^sadtalker '; then conda create -n sadtalker python=3.8 -y; fi
source /content/miniconda3/bin/activate sadtalker
python -m pip install -q --upgrade 'pip<25'
python -m pip install -q 'torch==2.0.1' 'torchvision==0.15.2' 'torchaudio==2.0.2' --index-url https://download.pytorch.org/whl/cu118
python -m pip install -q -r /content/SadTalker/requirements.txt
sed -i 's/from torchvision.transforms.functional_tensor import rgb_to_grayscale/from torchvision.transforms.functional import rgb_to_grayscale/' /content/miniconda3/envs/sadtalker/lib/python3.8/site-packages/basicsr/data/degradations.py 2>/dev/null || true
python -c 'import torch;print("CUDA:",torch.cuda.is_available())'
'''
subprocess.run(['bash','-lc',setup],check=True)
print('Dependencias listas')

In [ ]:
import subprocess
from pathlib import Path
subprocess.run(['bash','scripts/download_models.sh'],cwd='/content/SadTalker',check=True)
assert list(Path('/content/SadTalker/checkpoints').glob('*.safetensors')),'No se descargaron los pesos neuronales'
print('Pesos del motor listos')

## Generación neuronal del rostro

Este paso produce el MP4 de AURA con animación facial basada en audio. No es la animación aproximada de fotogramas 2D de la aplicación.

In [ ]:
import subprocess
cmd=['/content/miniconda3/bin/conda','run','--no-capture-output','-n','sadtalker','python','inference.py','--driven_audio','/content/AURA_media/voice.wav','--source_image','/content/AURA_media/aura_face.png','--result_dir','/content/AURA_neuronal','--preprocess','crop','--size','256','--expression_scale','1.0']
subprocess.run(cmd,cwd='/content/SadTalker',check=True)
print('Render neuronal finalizado')

In [ ]:
from pathlib import Path
from IPython.display import Video,display
from google.colab import files
videos=sorted(Path('/content/AURA_neuronal').rglob('*.mp4'),key=lambda p:p.stat().st_mtime)
assert videos,'No se creó el video. Revisá el registro de inferencia.'
result=videos[-1]
display(Video(filename=str(result),width=420,embed=True))
files.download(str(result))